# Recommender Systems — Python Examples

**Student version**

This notebook accompanies the introductory lecture on recommender systems.

We will implement:

1. **Content-based filtering**
2. **User-based collaborative filtering**
3. **Item-based collaborative filtering**
4. **Precision@K and Recall@K**
5. **Average Precision**

The goal is not to build a production recommender. The goal is to make the lecture concepts concrete and transparent.

A recurring question throughout the notebook is:

> **What exactly are we comparing?**

## 0. Setup

We use:

- `pandas` for tables
- `numpy` for numerical operations
- `scikit-learn` for cosine similarity in the **content-based** example

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.precision", 3)

# 1. Content-based filtering

A content-based recommender compares **item features** with a **user profile**.

Each movie below is described by simple binary genre features: `1` means the feature is present, `0` means it is not.

In [ ]:
movies = pd.DataFrame({
    "Sci-Fi":  [1, 1, 1, 1, 0, 0, 0],
    "Action":  [1, 0, 0, 1, 1, 0, 0],
    "Drama":   [0, 1, 1, 1, 0, 1, 0],
    "Romance": [0, 0, 0, 0, 0, 0, 1],
    "Comedy":  [0, 0, 0, 0, 0, 1, 1]
}, index=[
    "Matrix",
    "Dune",
    "Arrival",
    "Blade Runner 2049",
    "John Wick",
    "Barbie",
    "Mamma Mia"
])

movies

Karen likes **Matrix** and **Dune**.

A simple user profile can be created by averaging the features of the movies she likes.

In [ ]:
liked_movies = ["Matrix", "Dune"]

# TODO: create Karen's profile by averaging the feature values
# of the movies in liked_movies.
user_profile = ...

user_profile

Interpretation:

- Sci-Fi = 1.0 → strong evidence that Karen likes Sci-Fi
- Action = 0.5
- Drama = 0.5
- Romance / Comedy = 0.0 → **no evidence yet**, not necessarily a dislike

For this type of feature data, we use **cosine similarity** to compare Karen's profile with the candidate movies.

In [ ]:
# TODO: calculate cosine similarity between Karen's profile and every movie.
# Hint: cosine_similarity expects 2D input.

scores = ...

recommendations = pd.Series(scores, index=movies.index)

# Do not recommend movies Karen already watched.
recommendations = recommendations.drop(liked_movies)

recommendations.sort_values(ascending=False)

### Key idea

Content-based filtering recommends items because their **features are similar** to the features of items the user already likes.

It does **not** need other users' ratings.

# 2. Collaborative filtering data

Collaborative filtering uses **user-item interactions** instead of item features.

The ratings matrix below matches the lecture example. `NaN` means **unknown / not rated**.

> Missing rating ≠ rating of 0

In [ ]:
ratings = pd.DataFrame({
    "Matrix":     [5, 5, 1, 4],
    "Dune":       [5, 4, 1, 5],
    "Barbie":     [np.nan, 1, 5, 1],
    "Shrek":      [2, 2, 4, 2],
    "Arrival":    [np.nan, 5, 1, 5],
    "Mamma Mia":  [1, 1, 5, np.nan]
}, index=[
    "Karen",
    "Bob",
    "Sam",
    "Chloe"
])

ratings

## Comparing explicit ratings

For the 1–5 star ratings in this notebook, we use a deliberately simple comparison:

> **Average absolute rating difference on items both users/items have ratings for.**

For example, if two users differ by `0`, `1`, `3`, and `0` stars on four common movies, their average difference is `(0 + 1 + 3 + 0) / 4 = 1.0`.

**Smaller difference = more similar.**

This measure is chosen for transparency in this introductory notebook. It is not the only possible similarity measure.

In [ ]:
def average_rating_difference(a, b):
    """
    Return the mean absolute difference using only positions
    observed in BOTH Series. Lower means more similar.
    Q: Why don't we replace missing values with 0? What would happen if we did that?
    """
    # TODO 1: find positions where both a and b have a rating
    common = ...


    # TODO 2: keep only the common ratings
    x = ...
    y = ...

    # TODO 3: handle the case where there is no overlap
    if len(x) == 0:
        return np.nan

    # TODO 4: return the average absolute difference
    return ...

# 3. User-based collaborative filtering

For user-based collaborative filtering:

> Find users who gave **similar ratings** to the target user.

We compare **rows** of the ratings matrix.

First compare **Karen** and **Bob** using only movies they both rated.

In [ ]:
karen_bob_common = pd.concat(
    [ratings.loc["Karen"], ratings.loc["Bob"]],
    axis=1
).dropna()
karen_bob_common.columns = ["Karen", "Bob"]

karen_bob_common

In [ ]:
# TODO: calculate the average rating difference between Karen and Bob.
karen_bob_difference = ...

karen_bob_difference

Now calculate Karen's average rating difference from **every other user**.

Remember: **lower = more similar**.

In [ ]:
# TODO: calculate Karen's difference from Bob, Sam, and Chloe.
# Store the results in a dictionary called user_differences.

user_differences = {
    # "Bob": ...,
    # "Sam": ...,
    # "Chloe": ...
}

pd.Series(user_differences).sort_values()

### Predict Karen's rating for an unseen movie

A simple **k-nearest-neighbours** approach:

1. Keep users who rated the target movie.
2. Find which of those users are most similar to Karen.
3. Take the `k` nearest neighbours.
4. Average their ratings for the target movie.

For this teaching example we use `k = 2`.

In [ ]:
def predict_user_based(ratings,target_user, target_item, k=2):
    # TODO 1: find other users who rated target_item
    candidates = ...

    # TODO 2: calculate each candidate user's difference from target_user
    differences = ...

    # TODO 3: take the k users with the SMALLEST difference
    nearest = ...

    # TODO 4: average their ratings for target_item
    return ...

In [ ]:
# TODO: predict Karen's rating for Arrival and Barbie.
arrival_prediction_user = ...
barbie_prediction_user = ...

print("Predicted Karen rating for Arrival:", arrival_prediction_user)
print("Predicted Karen rating for Barbie: ", barbie_prediction_user)

### Key idea

**User-based CF:** compare **rows**.

> Who rated movies similarly to Karen, and what did those users think of movies Karen has not rated yet?

# 4. Item-based collaborative filtering

For item-based collaborative filtering:

> Find items that receive **similar ratings from users**.

We now compare **columns** instead of rows. The same average-rating-difference function can be reused.

Start with **Dune**. Which movies were rated most similarly to Dune?

In [ ]:
# TODO: compare Dune with every other movie.
# Store the average rating differences in item_differences_from_dune.

item_differences_from_dune = {
    # movie: average_rating_difference(ratings["Dune"], ratings[movie])
}

pd.Series(item_differences_from_dune).sort_values()

For this dataset, **Arrival** and **Matrix** are treated most similarly to Dune.

Important distinction:

- **Content-based:** Dune is similar to Arrival because their **features** are similar.
- **Item-based CF:** Dune is similar to Arrival because **users rated them similarly**.

## Predict an unseen rating with item-based CF

To predict Karen's rating for an unseen movie:

1. Compare the candidate movie with movies Karen already rated.
2. Find the `k` most similar rated movies (smallest average difference).
3. Average Karen's own ratings for those neighbouring movies.

Again, we use `k = 2` for this small teaching example.

In [ ]:
def predict_item_based(ratings,target_user, target_item, k=2):
    user_ratings = ratings.loc[target_user]

    # TODO 1: find movies target_user has already rated
    rated_items = ...

    # TODO 2: compare target_item with each of those rated items
    differences = ...

    # TODO 3: choose the k items with the SMALLEST difference
    nearest_items = ...

    # TODO 4: average target_user's ratings for those nearest items
    return ...

In [ ]:
# TODO: predict Karen's rating for Arrival and Barbie.
arrival_prediction_item = ...
barbie_prediction_item = ...

print("Predicted Karen rating for Arrival:", arrival_prediction_item)
print("Predicted Karen rating for Barbie: ", barbie_prediction_item)

### Inspect the Barbie prediction

For Barbie, the nearest items among Karen's already-rated movies should be **Mamma Mia** and **Shrek**.

Karen rated them `1` and `2`, so the simple item-based prediction is `(1 + 2) / 2 = 1.5`.

In [ ]:
# TODO: verify which already-rated items are closest to Barbie.
# Produce a sorted Series of average rating differences.

barbie_item_differences = ...
barbie_item_differences

### Important limitation

With tiny data, a low difference may be based on only a few common ratings. For example, Barbie and Mamma Mia are compared using only the users who rated **both** movies.

Real recommender systems normally need more safeguards than this notebook uses. Here we deliberately keep the calculation simple so the collaborative-filtering logic remains visible.

# 5. Precision@K and Recall@K

Suppose the relevant movies for a user are `B`, `D`, and `F`, and the recommender returns:

`B, A, D, C, F`

In [ ]:
relevant = {"B", "D", "F"}
recommendations = ["B", "A", "D", "C", "F"]

In [ ]:
def precision_at_k(recommendations, relevant, k):
    # TODO: return the fraction of the top-k recommendations that are relevant
    ...


def recall_at_k(recommendations, relevant, k):
    # TODO: return the fraction of all relevant items retrieved in the top-k
    ...

In [ ]:
# TODO: calculate Precision@5 and Recall@5
print("Precision@5:", ...)
print("Recall@5:   ", ...)

Interpretation:

- **Precision@5 = 0.6** → 3 of the 5 recommended items were relevant.
- **Recall@5 = 1.0** → all 3 relevant items were retrieved.

But Precision@K and Recall@K do not distinguish well between rankings that contain the same relevant items at different positions.

# 6. Average Precision

Compare two rankings:

**Model A:** `B, A, D, C, F`  
**Model B:** `A, C, B, D, F`

Both retrieve the same three relevant items, but Model A places relevant items earlier.

In [ ]:
model_a = ["B", "A", "D", "C", "F"]
model_b = ["A", "C", "B", "D", "F"]
relevant = {"B", "D", "F"}

In [ ]:
def average_precision(recommendations, relevant):
    # TODO: calculate precision each time a relevant item is encountered,
    # then average those precision values over all relevant items.
    ...

In [ ]:
# TODO: calculate AP for both models
print("AP Model A:", ...)
print("AP Model B:", ...)

Expected interpretation:

- Model A ≈ **0.756**
- Model B ≈ **0.478**

Average Precision rewards rankings where relevant items appear **earlier**.

# 7. Summary

| Concept | What is compared? | Method used here |
|---|---|---|
| Content-based | User profile vs. item features | Cosine similarity |
| User-based CF | User rating rows | Average absolute rating difference |
| Item-based CF | Item rating columns | Average absolute rating difference |
| Precision@K | Top-K recommendations vs. relevant items | Hit fraction in Top-K |
| Recall@K | Retrieved relevant items vs. all relevant items | Coverage of relevant set |
| Average Precision | Positions of relevant items in the ranking | Precision at each hit |

## Final reflection

For each recommender system, ask:

1. What data are we using?
2. What exactly are we comparing?
3. How are we deciding what is similar?
4. How do we produce a predicted score or ranking?
5. How do we evaluate whether that ranking is useful?

## Final check

Without looking back, explain in your own words:

1. Why is cosine similarity used for the **content-feature** example but not for the explicit-rating CF examples in this notebook?
2. What is the difference between **user-based** and **item-based** collaborative filtering?
3. Why do we ignore missing ratings rather than replacing them with zero?
4. Why can Precision@K and Recall@K fail to distinguish two useful rankings?